# r2-001 Problem 5 — Reference solution (two Lorentzian peaks)

One reference approach: two cheap initializers per signal — a learned amortized regressor (MLP trained on training plus simulated signals) and a closed-form greedy peak picker — each polished by per-signal least squares through the differentiable forward model; for every signal the fit with the smaller residual is kept.
The cell below is the problem notebook's supplied setup cell, unchanged.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

f_train, theta_train = r2_001_data.train_rows("lorentz")
f_val, theta_val = r2_001_data.val_rows("lorentz")
print(f_train.shape, theta_train.shape, f_val.shape)
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["lorentz"])

## Budget and the committed baseline

In [ ]:
budget = r2_001_data.StepBudget(2000)
X_GRID = r2_001_data.LORENTZ_X.double()
DX = 1.0 / 40

baseline = r2_001_data.baseline_score("lorentz")
print("baseline under the protocol:", baseline)

## Approach

In [ ]:
SIM_SEED, SIM_ROWS, HIDDEN, NET_STEPS, BATCH = 1, 20000, 128, 700, 512
REFINE_STEPS = 200
SCALE = torch.tensor([1.0, 10.0, 30.0, 1.0, 10.0, 30.0])     # puts a, c, w errors on comparable scales

seam = set()
state = {}


def lorentz(x, a, c, w):
    return a / (1.0 + ((x - c) / w) ** 2)


def greedy_init(f):
    """Closed-form initializer: take the tallest sample, estimate (a, c, w), subtract that peak, repeat."""
    r = f.double().clone()
    out = torch.zeros(len(r), 6, dtype=torch.float64)
    for k in range(2):
        j = r.argmax(dim=1)
        a = r.gather(1, j[:, None])[:, 0].clamp_min(0.3)
        jm, jp = (j - 1).clamp(0, 39), (j + 1).clamp(0, 39)
        ym, y0, yp = (r.gather(1, t[:, None])[:, 0] for t in (jm, j, jp))
        den = ym - 2 * y0 + yp
        shift = torch.where(den.abs() > 1e-9, 0.5 * (ym - yp) / den, torch.zeros_like(den)).clamp(-0.5, 0.5)
        c = X_GRID[j] + shift * DX                                   # parabolic sub-sample centre
        window = (j[:, None] + torch.arange(-3, 4)[None]).clamp(0, 39)
        area = r.gather(1, window).clamp_min(0).sum(1) * DX          # area of a Lorentzian = pi a w
        w = (area / (math.pi * a)).clamp(0.02, 0.08)
        out[:, 3 * k:3 * k + 3] = torch.stack([a, c, w], dim=1)
        r = r - lorentz(X_GRID[None], a[:, None], c[:, None], w[:, None])
    return out


def fit(train_x, train_y):
    for row in train_x:
        seam.add(r2_001_data.split_of("lorentz", row))
    mean, std = train_x.mean(0), train_x.std(0)
    torch.manual_seed(SEED)
    net = nn.Sequential(nn.Linear(40, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, 6))
    base_opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(base_opt, max_lr=3e-3, total_steps=NET_STEPS)
    opt = budget.wrap(base_opt)
    xn, targets = (train_x - mean) / std, train_y * SCALE
    gen = torch.Generator().manual_seed(SEED)
    for _ in range(NET_STEPS):
        idx = torch.randint(0, len(xn), (BATCH,), generator=gen)
        pred, tgt = net(xn[idx]), targets[idx]
        same = ((pred - tgt) ** 2).sum(1)
        swap = ((pred[:, [3, 4, 5, 0, 1, 2]] - tgt) ** 2).sum(1)
        loss = torch.minimum(same, swap).mean()                      # permutation-invariant loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
    state.update(net=net.eval(), mean=mean, std=std)
    print(f"amortized regressor trained: {NET_STEPS} steps, last mini-batch loss {loss.item():.5f}")


def learned_init(f):
    with torch.no_grad():
        p = (state["net"]((f - state["mean"]) / state["std"]) / SCALE).double()
    p[:, [0, 3]] = p[:, [0, 3]].clamp_min(0.1)
    p[:, [2, 5]] = p[:, [2, 5]].clamp(0.015, 0.1)
    return p


def refine(f, init, lr):
    return r2_001_data.lorentz_ls_fit(f, init, steps=REFINE_STEPS, lr=lr, optimizer_wrapper=budget.wrap)


def residual(params, f):
    return ((r2_001_data.lorentz_values(X_GRID, params.double()) - f.double()) ** 2).mean(dim=1)


def best_of(fits, f):
    res = torch.stack([residual(p, f) for p in fits])                # (n_fits, N)
    pick = res.argmin(dim=0)
    return torch.stack(fits)[pick, torch.arange(f.shape[0])]


def predict(f):
    fits = [refine(f, learned_init(f), state["lr"]), refine(f, greedy_init(f), state["lr"])]
    return best_of(fits, f)


x_sim, y_sim = r2_001_data.simulate("lorentz", SIM_ROWS, seed=SIM_SEED)
fit(torch.cat([f_train, x_sim]), torch.cat([theta_train, y_sim]))
assert seam <= {"train", None} and "val" not in seam and "test" not in seam
print("seam:", seam, "| steps used after fit:", budget.used)

## Validation evidence (every choice is made here)

In [ ]:
def val_err(pred):
    return float(r2_001_data.pair_parameter_error_per_example(pred, theta_val).mean())


init_net, init_greedy = learned_init(f_val), greedy_init(f_val)
fit_net = {lr: refine(f_val, init_net, lr) for lr in (0.01, 0.02)}
fit_greedy = {lr: refine(f_val, init_greedy, lr) for lr in (0.01, 0.02)}
rows = {
    "learned amortized regressor alone (no refinement)": val_err(init_net),
    "greedy closed-form peak picker alone (no refinement)": val_err(init_greedy),
}
for lr in (0.01, 0.02):
    rows[f"learned init + {REFINE_STEPS} LS steps, lr {lr}"] = val_err(fit_net[lr])
    rows[f"greedy init + {REFINE_STEPS} LS steps, lr {lr}"] = val_err(fit_greedy[lr])
    rows[f"best-of-two by residual, lr {lr}"] = val_err(best_of([fit_net[lr], fit_greedy[lr]], f_val))
for name, value in rows.items():
    print(f"val error, {name:55s} {value:.5f}")
state["lr"] = min((0.01, 0.02), key=lambda lr: rows[f"best-of-two by residual, lr {lr}"])
print("chosen refinement lr:", state["lr"])
print("steps used before the test call:", budget.used, "of", budget.max_steps)

## The single locked-test call

In [ ]:
assert r2_001_data.test_call_count("lorentz") == 0
result = r2_001_data.final_test_score(predict, n_boot=1000, alpha=0.05, task="lorentz")
assert r2_001_data.test_call_count("lorentz") == 1
elapsed = budget.elapsed()
print(result)
print(f"budget: {budget.used} of {budget.max_steps} steps, elapsed {elapsed:.1f} s")

## Writeup

**Approach.** With i.i.d. Gaussian noise ($\sigma=0.02$) the maximum-likelihood estimate of the six parameters is the per-signal least-squares fit through the exact forward model `lorentz_values`; the difficulty is only the non-convex landscape (peaks can be swapped, merged, or one peak can grow wide to cover both), which is why the fixed-start baseline fails on many signals.
I give every signal two independent starting points and refine both by per-signal least squares (`lorentz_ls_fit`, Adam on amplitudes, centres, and log-widths, 200 steps, lr chosen on validation): (i) a **learned amortized regressor** — an MLP (40 standardized samples → 128 → 128 → 6, GELU) trained for 700 Adam steps on the 800 training signals plus 20,000 signals from `simulate("lorentz", 20000, seed=1)` with a permutation-invariant squared loss on scaled parameters; and (ii) a **closed-form greedy peak picker** — tallest sample, parabolic sub-sample centre, width from the local area ($\int L = \pi a w$), subtract, repeat.
For each signal the refined fit with the smaller mean squared residual is kept; this uses only the signal itself, so it is a legitimate per-signal model-selection rule.

**Alternatives considered** (all on the 200 validation signals).
(1) *Learned amortized regressor alone*, the learned approach without any per-signal fitting: 0.05056, an order of magnitude worse than any refined fit — with 700 CPU steps it locates the peaks roughly but cannot resolve widths and amplitudes to the noise level.
(2) *Greedy peak picker + least squares* (a purely non-learned pipeline): 0.00597 at lr 0.02 (0.00578 at lr 0.01), versus 0.00504 for the chosen best-of-two; (3) *learned init + least squares* alone: 0.00592 at lr 0.02 (0.00727 at lr 0.01).
Each single-start pipeline loses on a few signals where its start sits in a wrong basin, and the two starts fail on different signals, which is why selecting by residual helps; between learning rates the best-of-two scores 0.00505 (lr 0.01) and 0.00504 (lr 0.02), so lr 0.02 was chosen.

**Evaluation.** All selection used validation only; the locked test set was scored once with `final_test_score` (n = 200, 1,000 bootstrap resamples, α = 0.05).
Test score **0.00527** (mean permutation-invariant parameter error), 95% bootstrap interval [0.00484, 0.00568]; the committed baseline under the same protocol scores 0.01711 [0.00981, 0.02607], so the improvement is well outside both intervals and clears the full-credit threshold 0.008231.
Budget: 1,900 of 2,000 optimizer steps (700 regressor, 800 validation refinements, 400 test refinements), about 7 s by `budget.elapsed()` on this CPU.

**Limitations.** When the two peaks are close (separation near the minimum $1.5(w_1+w_2)$) and one is much weaker, both starts can converge to a single wide peak plus a spurious small one; the residual rule then picks the less bad of two wrong fits.
I would test this by simulating fresh signals, binning the error by separation in units of $w_1+w_2$ and by amplitude ratio, and checking whether the selected residual s.d. exceeds the noise level $0.02$ on the failing bins; a third start (for example the greedy picker with the two detected peaks' roles exchanged) would be the first remedy to validate.
Widths are also the least identifiable parameters at a 0.025 sampling step when $w\approx0.02$, so they would dominate the error in that regime.
Finally, `simulate("lorentz", n, seed)` raises "rounding broke the separation rule" for some seeds at large `n` (seed 7 fails even at n = 2,000), so the simulated-data seed is fixed at 1.

### Answer check

In [ ]:
assert r2_001_data.test_call_count("lorentz") == 1, "final_test_score must be called exactly once"
assert seam <= {"train", None}
assert budget.used <= 2000 and elapsed < 60
assert result.n_examples == 200 and result.ci_low <= result.score <= result.ci_high
assert result.score < baseline.score
assert result.score <= 0.008231, "full-credit tier: score <= R + 0.25 g"
print(f"Problem 5 test score: {result.score:.5f} (95% CI {result.ci_low:.5f}-{result.ci_high:.5f}, n = {result.n_examples}); "
      f"baseline {baseline.score:.5f}")